In [ ]:
import openai
import re
import httpx
import os
from dotenv import load_dotenv

_ = load_dotenv()
from openai import OpenAI
import os
from urllib.request import urlopen, Request
import base64
from pathlib import Path
from dotenv import load_dotenv

# Add references
from openai import OpenAI
from azure.identity import DefaultAzureCredential, get_bearer_token_provider

In [ ]:
# Get configuration settings 
load_dotenv()
openai_endpoint = os.getenv("ENDPOINT")
model_deployment =  os.getenv("MODEL_DEPLOYMENT")


# Create an OpenAI client
credential = DefaultAzureCredential()
token_provider = get_bearer_token_provider(credential, "https://ai.azure.com/.default")
client = OpenAI(
    base_url=openai_endpoint,
    api_key=token_provider()
) 
chat_completion = client.chat.completions.create(
    model=model_deployment,
    messages=[{"role": "user", "content": "who is prime minester of india"}]
) 
print(chat_completion.choices[0].message.content)  

In [ ]:
import nest_asyncio
nest_asyncio.apply()

In [ ]:
from llama_index.core.tools import FunctionTool

def add(x: int, y: int) -> int:
    """Adds two integers together."""
    return x + y

def mystery(x: int, y: int) -> int: 
    """Mystery function that operates on top of two numbers."""
    return (x + y) * (x + y)


add_tool = FunctionTool.from_defaults(fn=add)
mystery_tool = FunctionTool.from_defaults(fn=mystery)

In [ ]:
llm = OpenAI(model=model_deployment,
             api_base=openai_endpoint,
             api_key=token_provider())

In [ ]:
llm.complete("hello")

In [ ]:
from llama_index.llms.openai import OpenAI


response = llm.predict_and_call(
    [add_tool, mystery_tool], 
    "Tell me the output of the mystery function on 2 and 9", 
    verbose=True
)
print(str(response))

In [ ]:
from llama_index.core import SimpleDirectoryReader
documents = SimpleDirectoryReader(
    input_files=[r"C:\Users\a855099\Documents\GitHub\mslearn-ai-vision\Labfiles\gen-ai-vision\India Insurance - Group & Voluntary Insurance Policy 2026-27.pdf"]
).load_data()


In [ ]:
from llama_index.core.node_parser import SentenceSplitter
splitter = SentenceSplitter(chunk_size=1024)
nodes = splitter.get_nodes_from_documents(documents)

In [33]:
print(nodes[0].get_content(metadata_mode="all"))

file_path: C:\Users\a855099\Documents\GitHub\mslearn-ai-vision\Labfiles\gen-ai-vision\India Insurance - Group & Voluntary Insurance Policy 2026-27.pdf
file_name: India Insurance - Group & Voluntary Insurance Policy 2026-27.pdf
file_type: application/pdf
file_size: 942529
creation_date: 2026-09-26
last_modified_date: 2026-08-05

%PDF-1.7

4 0 obj
(Identity)
endobj
5 0 obj
(Adobe)
endobj
8 0 obj
<<
/Filter /FlateDecode
/Length 34916
/Length1 72552
/Type /Stream
>>
stream
9ߋ4sX1޿»']2?;O L"=h(Ƽ&.ΈQc+R9#˘_u5z'F20clԇ7eFC+jr{y3x	6W)wn./zgS*MƲCZW\U4n,;m.+5XزՌzW;`h;:	hk5A*27Q>Ȣ!_0_o1׷~O/m){93ۑѱׅ.M1)/sd.|%Yߗ։.)X+y(k3ЫѬv3-shb~S<ſӍiD虸y\WdpcL8״\S&I/Ef/۬4ngk[5os&<Wef޹<0]:>|swܯ<5I-o6m;^u2wYo5,hK>lҏ#Uk\~KϘyiݤ='hfz;YƬ_Gv?#}ch-#eﯙmF}./AdOs5O_\l?HV6ZcLIWIu,C`
2,eٚV'%a
1M˘ϙv.c길Ɖ;`1xǨ}?jf}߱Z[x&2#xo}궳#X.p,>a4Vǖe[om;ցgAF+7O&<?rŪ~׊/V%_ߥ>3DџEOpa#h6履51v=/L%'~~~>B?K_opMehf8dSՙF|<gg67b 3EB!鈄{bUgL!_^.t_z\(FHv&)@Up'?

In [23]:
from llama_index.core import Settings
from llama_index.llms.azure_openai import AzureOpenAI
from llama_index.embeddings.azure_openai import AzureOpenAIEmbedding

Settings.llm = AzureOpenAI(engine=model_deployment,
                           model="gpt-5-mini",
                           azure_endpoint=openai_endpoint.replace("/openai/v1", ""),
                           azure_ad_token_provider=token_provider,
                           use_azure_ad=True,
                           api_version="2024-06-01"
                           )

In [25]:
embedding_deployment = os.getenv("EMBEDDING_MODEL_DEPLOYMENT")

In [26]:
Settings.embed_model = AzureOpenAIEmbedding(azure_deployment=embedding_deployment,
                                            model="text-embedding-3-small",
                                            azure_endpoint=openai_endpoint.replace("/openai/v1", ""),
                                            azure_ad_token_provider=token_provider,
                                            use_azure_ad=True,
                                            api_version="2024-06-01"
                                       )

In [40]:
from llama_index.core import VectorStoreIndex

vector_index = VectorStoreIndex(nodes)
query_engine = vector_index.as_query_engine(similarity_top_k=2)

In [49]:
from llama_index.core.vector_stores import MetadataFilters

query_engine = vector_index.as_query_engine(
    similarity_top_k=2,
    filters=MetadataFilters.from_dicts(
        [
            {"key": "page_label", "value": "1"}
        ]
    )
)

response = query_engine.query(
    "what is Limit of Coverage applicable", 
)

AzureCliCredential.get_token_info failed: Failed to invoke the Azure CLI


CredentialUnavailableError: Failed to invoke the Azure CLI

In [45]:
print(response)

Empty Response


In [30]:
for n in response.source_nodes:
    print(n.metadata)

In [46]:
from typing import List
from llama_index.core.vector_stores import FilterCondition


def vector_query(
    query: str, 
    page_numbers: List[str]
) -> str:
    """Perform a vector search over an index.
    
    query (str): the string query to be embedded.
    page_numbers (List[str]): Filter by set of pages. Leave BLANK if we want to perform a vector search
        over all pages. Otherwise, filter by the set of specified pages.
    
    """

    metadata_dicts = [
        {"key": "page_label", "value": p} for p in page_numbers
    ]
    
    query_engine = vector_index.as_query_engine(
        similarity_top_k=2,
        filters=MetadataFilters.from_dicts(
            metadata_dicts,
            condition=FilterCondition.OR
        )
    )
    response = query_engine.query(query)
    return response
    

vector_query_tool = FunctionTool.from_defaults(
    name="vector_tool",
    fn=vector_query
)

In [47]:
llm = OpenAI(model=model_deployment,
             api_base=openai_endpoint,
             api_key=token_provider(),
             temperature=0)
response = llm.predict_and_call(
    [vector_query_tool], 
    "What are the high-level results of indian insurance as described on page 2?", 
    verbose=True
)

=== Calling Function ===
Calling function: vector_tool with args: {"query": "high-level results of Indian insurance", "page_numbers": ["2"]}
=== Function Output ===
Empty Response


In [48]:
for n in response.source_nodes:
    print(n.metadata)